# Mixture of Experts: How Routing Becomes GPU Computation

This is the step-by-step companion to [Mixture of Experts: How Routing Becomes GPU Computation](https://g-u-n.github.io/blogs/mixture-of-experts.html).

**Open this notebook in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/G-U-N/G-U-N.github.io/blob/master/blogs/code/mixture_of_experts_colab.ipynb)

We start with an ordinary GEMM, turn router decisions into an expert-major packed layout, and write the grouped GEMM that consumes that layout. The notebook checks the SwiGLU MoE forward and fixed-top-k backward against a PyTorch reference, including the selected-softmax router path. We then test Split-K and column-major scheduling for decode. Optional cells show how the same layout is passed to DeepGEMM's official BF16 grouped API on a compatible Hopper or Blackwell GPU.

This is a single-GPU teaching notebook. For multi-GPU dispatch/combine, use the companion [EP script](https://github.com/G-U-N/G-U-N.github.io/blob/master/blogs/code/moe_ep_hopper.py).

## 0. Setup: choose the GPU and data type

Regular Triton cells work on CUDA GPUs supported by the installed Triton version. We use BF16 on Ampere or newer and fall back to FP16 on a Colab T4. DeepGEMM is a separate guarded path. Its official README requires SM90 or SM100, CUDA 12.3+ for SM90, CUDA 12.9+ for SM100, PyTorch 2.1+, C++20, and CUTLASS. A usual Colab T4 or L4 runtime should therefore run the teaching kernels and skip the DeepGEMM cells.

In [ ]:
import importlib.util
import math
import statistics
import time
import subprocess
import sys
import torch
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. In Colab choose Runtime > Change runtime type > GPU.")

try:
    import triton
    import triton.language as tl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton"])
    import triton
    import triton.language as tl

DEVICE = torch.device("cuda")
CAPABILITY = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if CAPABILITY[0] >= 8 else torch.float16
print("device:", torch.cuda.get_device_name())
print("compute capability:", f"sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print("torch:", torch.__version__, "triton:", triton.__version__, "teaching dtype:", DTYPE)

def check_close(name, actual, expected, *, atol=2e-3, rtol=3e-2):
    assert actual.shape == expected.shape
    af, ef = actual.detach().float(), expected.detach().float()
    assert torch.isfinite(af).all() and torch.isfinite(ef).all(), f"{name}: non-finite values"
    if af.numel() == 0:
        print(f"{name}: ok (empty expert)")
        return
    diff = af - ef
    rel_l2 = diff.norm() / ef.norm().clamp_min(1e-12)
    assert rel_l2 <= rtol, f"{name}: relative L2 error {rel_l2.item():.3e} > {rtol}"
    torch.testing.assert_close(af, ef, atol=atol, rtol=rtol)
    print(f"{name}: ok (max abs {diff.abs().max().item():.3e}, relative L2 {rel_l2.item():.3e})")

def benchmark_wall_ms(fn, *, warmup=5, iters=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(iters):
        start = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        samples.append((time.perf_counter() - start) * 1000)
    return statistics.median(samples)

def benchmark_ms(fn, *, warmup=5, iters=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    samples = []
    for _ in range(iters):
        start.record()
        fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end))
    return statistics.median(samples)

## 1. GEMM: compute an FFN projection

For \(A\in\mathbb{R}^{M\times K}\) and \(B\in\mathbb{R}^{K\times N}\), \(C=AB\) has shape \(M\times N\). A Triton program owns a tile of rows and columns and reduces over \(K\). An MoE layer changes how rows are grouped, not this matrix product.

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32}, num_warps=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 32}, num_warps=8),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def _matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    pid_m, pid_n = pid // num_pid_n, pid % num_pid_n
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for start_k in range(0, K, BLOCK_K):
        ks = start_k + tl.arange(0, BLOCK_K)
        a = tl.load(a_ptr + rows[:, None] * stride_am + ks[None, :] * stride_ak,
                    mask=(rows[:, None] < M) & (ks[None, :] < K), other=0.0)
        b = tl.load(b_ptr + ks[:, None] * stride_bk + cols[None, :] * stride_bn,
                    mask=(ks[:, None] < K) & (cols[None, :] < N), other=0.0)
        acc += tl.dot(a, b)
    tl.store(c_ptr + rows[:, None] * stride_cm + cols[None, :] * stride_cn,
             acc.to(c_ptr.dtype.element_ty),
             mask=(rows[:, None] < M) & (cols[None, :] < N))

def triton_matmul(a, b):
    assert a.is_cuda and b.is_cuda and a.dtype == b.dtype == DTYPE
    assert a.ndim == b.ndim == 2 and a.shape[1] == b.shape[0]
    a, b = a.contiguous(), b.contiguous()
    out = torch.empty((a.shape[0], b.shape[1]), device=a.device, dtype=a.dtype)
    grid = lambda META: (triton.cdiv(a.shape[0], META["BLOCK_M"]) * triton.cdiv(b.shape[1], META["BLOCK_N"]),)
    _matmul_kernel[grid](
        a, b, out, a.shape[0], b.shape[1], a.shape[1],
        a.stride(0), a.stride(1), b.stride(0), b.stride(1), out.stride(0), out.stride(1),
    )
    return out

M, K, N = 127, 96, 80
a = torch.randn((M, K), device=DEVICE, dtype=DTYPE)
b = torch.randn((K, N), device=DEVICE, dtype=DTYPE)
check_close("Triton GEMM", triton_matmul(a, b), a @ b)

## 2. Routing: group tokens by expert

We choose top-k expert indices from the router logits, then apply softmax only to the selected logits, as in Equation (2) of the article. We expand selected token–expert pairs, sort them by expert, and store the rows in expert-major order. offsets[e:e+2] gives the segment for expert e. The selected weights remain differentiable; the top-k expert indices are discrete.

In [ ]:
def build_packed(x, selected_expert, selected_weight, num_experts):
    T, top_k = selected_expert.shape
    token_ids = torch.arange(T, device=x.device).repeat_interleave(top_k)
    expert_ids = selected_expert.reshape(-1)
    pair_weight = selected_weight.reshape(-1)
    order = torch.argsort(expert_ids, stable=True)
    token_ids, expert_ids = token_ids[order], expert_ids[order]
    pair_weight = pair_weight[order]
    counts = torch.bincount(expert_ids, minlength=num_experts)
    offsets = torch.cat([
        torch.zeros(1, device=x.device, dtype=torch.int32),
        counts.cumsum(0).to(torch.int32),
    ])
    return token_ids, expert_ids, pair_weight, order, offsets, counts

def route_and_pack(x, router_logits, top_k):
    selected_expert = torch.topk(router_logits, top_k, dim=-1).indices
    selected_logits = router_logits.gather(1, selected_expert)
    selected_weight = torch.softmax(selected_logits, dim=-1)
    packed = build_packed(x, selected_expert, selected_weight, router_logits.shape[-1])
    return selected_expert, selected_weight, packed

T, D, E, TOP_K = 24, 32, 6, 2
x = torch.randn((T, D), device=DEVICE, dtype=DTYPE)
router_logits = torch.randn((T, E), device=DEVICE, dtype=torch.float32)
selected_expert, selected_weight, packed = route_and_pack(x, router_logits, TOP_K)
print("token-major x:", tuple(x.shape))
print("expert-major rows:", tuple(packed[0].shape))
print("expert counts:", packed[-1].tolist())
print("offsets:", packed[-2].tolist())

## 3. Grouped GEMM: schedule uneven expert batches

All experts have the same K,N, but each expert has a different M_e. We create a row tile for each BLOCK_M rows of a non-empty expert segment. Each program reads its expert id and tile-row id, computes the segment offset, and executes an ordinary tiled GEMM. This groups the M dimension as in DeepGEMM's contiguous MoE API. Our explicit tile list is a simple teaching scheduler; Triton's official Group GEMM tutorial uses a persistent scheduler.

In [ ]:
def make_contiguous_schedule(counts, block_m, *, device):
    expert_tiles, row_tiles = [], []
    for expert, count in enumerate(counts.tolist()):
        for row_tile in range((count + block_m - 1) // block_m):
            expert_tiles.append(expert)
            row_tiles.append(row_tile)
    return (
        torch.tensor(expert_tiles, device=device, dtype=torch.int32),
        torch.tensor(row_tiles, device=device, dtype=torch.int32),
    )

@triton.jit
def _m_grouped_nt_contiguous(
    a_ptr, b_ptr, c_ptr, offsets_ptr, tile_expert_ptr, tile_m_ptr,
    N: tl.constexpr, K: tl.constexpr,
    stride_am, stride_ak, stride_be, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    expert = tl.load(tile_expert_ptr + pid_m)
    tile_m = tl.load(tile_m_ptr + pid_m)
    start, end = tl.load(offsets_ptr + expert), tl.load(offsets_ptr + expert + 1)
    rows = tile_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for start_k in range(0, K, BLOCK_K):
        ks = start_k + tl.arange(0, BLOCK_K)
        a = tl.load(a_ptr + (start + rows)[:, None] * stride_am + ks[None, :] * stride_ak,
                    mask=(start + rows[:, None] < end) & (ks[None, :] < K), other=0.0)
        b = tl.load(b_ptr + expert * stride_be + ks[:, None] * stride_bk + cols[None, :] * stride_bn,
                    mask=(ks[:, None] < K) & (cols[None, :] < N), other=0.0)
        acc += tl.dot(a, b)
    tl.store(c_ptr + (start + rows)[:, None] * stride_cm + cols[None, :] * stride_cn,
             acc.to(c_ptr.dtype.element_ty),
             mask=(start + rows[:, None] < end) & (cols[None, :] < N))

def grouped_mm_contiguous(a, b, offsets, *, block_m=32, block_n=64, block_k=32,
                          schedule=None, out=None):
    assert a.ndim == 2 and b.ndim == 3 and b.shape[2] == a.shape[1]
    assert a.dtype == b.dtype == DTYPE
    a, b = a.contiguous(), b.contiguous()
    if schedule is None:
        counts = offsets[1:] - offsets[:-1]
        schedule = make_contiguous_schedule(counts.cpu(), block_m, device=a.device)
    tile_expert, tile_m = schedule
    if out is None:
        out = torch.empty((a.shape[0], b.shape[1]), device=a.device, dtype=a.dtype)
    if tile_expert.numel() == 0:
        return out
    grid = (tile_expert.numel(), triton.cdiv(b.shape[1], block_n))
    _m_grouped_nt_contiguous[grid](
        a, b, out, offsets, tile_expert, tile_m, b.shape[1], a.shape[1],
        a.stride(0), a.stride(1), b.stride(0), b.stride(2), b.stride(1),
        out.stride(0), out.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k, num_warps=4,
    )
    return out

def grouped_mm_reference(a, b, offsets):
    pieces = []
    for expert in range(b.shape[0]):
        start, end = offsets[expert].item(), offsets[expert + 1].item()
        pieces.append(a[start:end] @ b[expert].transpose(0, 1))
    return torch.cat(pieces, dim=0)

counts = torch.tensor([7, 0, 11, 3, 2, 1], device=DEVICE, dtype=torch.int32)
offsets = torch.cat([torch.zeros(1, device=DEVICE, dtype=torch.int32), counts.cumsum(0)])
packed_a = torch.randn((int(counts.sum()), K), device=DEVICE, dtype=DTYPE)
expert_b = torch.randn((E, N, K), device=DEVICE, dtype=DTYPE)
check_close("contiguous grouped GEMM", grouped_mm_contiguous(packed_a, expert_b, offsets),
            grouped_mm_reference(packed_a, expert_b, offsets))

## 4. Forward: compute and combine the selected experts

For each selected pair we compute \(g=xW_{\rm gate,e}^{\mathsf T}\), \(u=xW_{\rm up,e}^{\mathsf T}\), \(h=\operatorname{SiLU}(g)\odot u\), and \(y=hW_{\rm down,e}^{\mathsf T}\). These are three grouped GEMMs. We then multiply by the routing weight and scatter-add the pair outputs back to token-major order.

In [ ]:
def moe_reference(x, selected_expert, selected_weight, w_gate, w_up, w_down):
    token_ids, expert_ids, pair_weight, _, offsets, _ = build_packed(
        x, selected_expert, selected_weight, w_gate.shape[0]
    )
    a = x[token_ids]
    gate = torch.bmm(a[:, None, :], w_gate[expert_ids].transpose(1, 2)).squeeze(1)
    up = torch.bmm(a[:, None, :], w_up[expert_ids].transpose(1, 2)).squeeze(1)
    hidden = F.silu(gate) * up
    pair_out = torch.bmm(hidden[:, None, :], w_down[expert_ids].transpose(1, 2)).squeeze(1)
    out = torch.zeros(x.shape, device=x.device, dtype=torch.float32)
    out.index_add_(0, token_ids, pair_out.float() * pair_weight[:, None])
    return out.to(x.dtype)

def selected_expert_path_forward(
    x, selected_expert, selected_weight, w_gate, w_up, w_down
):
    # Pack token-expert pairs in expert order.
    T, top_k = selected_expert.shape
    token_ids = torch.arange(T, device=x.device).repeat_interleave(top_k)
    expert_ids = selected_expert.reshape(-1)
    order = torch.argsort(expert_ids, stable=True)
    token_ids, expert_ids = token_ids[order], expert_ids[order]
    pair_weight = selected_weight.reshape(-1)[order]
    counts = torch.bincount(expert_ids, minlength=w_gate.shape[0])
    offsets = torch.cat([
        torch.zeros(1, device=x.device, dtype=torch.int32),
        counts.cumsum(0).to(torch.int32),
    ])

    # Compute each expert's SwiGLU FFN.
    a = x[token_ids].contiguous()
    gate = grouped_mm_contiguous(a, w_gate, offsets)
    up = grouped_mm_contiguous(a, w_up, offsets)
    hidden = torch.nn.functional.silu(gate) * up
    pair_out = grouped_mm_contiguous(hidden, w_down, offsets)

    # Accumulate weighted expert outputs at their original token positions.
    out = torch.zeros(x.shape, device=x.device, dtype=torch.float32)
    out.index_add_(0, token_ids, pair_out.float() * pair_weight[:, None])
    cache = (
        token_ids, expert_ids, pair_weight, order, offsets,
        a, gate, up, hidden, pair_out,
    )
    return out.to(x.dtype), cache

T, D, FFE, E, TOP_K = 32, 48, 64, 8, 2
torch.manual_seed(0)
x = torch.randn((T, D), device=DEVICE, dtype=DTYPE)
logits = torch.randn((T, E), device=DEVICE, dtype=torch.float32)
selected_expert = torch.topk(logits, TOP_K, dim=-1).indices
selected_weight = logits.gather(1, selected_expert).softmax(-1)
w_gate = torch.randn((E, FFE, D), device=DEVICE, dtype=DTYPE) * 0.05
w_up = torch.randn((E, FFE, D), device=DEVICE, dtype=DTYPE) * 0.05
w_down = torch.randn((E, D, FFE), device=DEVICE, dtype=DTYPE) * 0.05
out_ref = moe_reference(x, selected_expert, selected_weight, w_gate, w_up, w_down)
out_tri, _ = selected_expert_path_forward(x, selected_expert, selected_weight, w_gate, w_up, w_down)
check_close("SwiGLU MoE forward", out_tri, out_ref, atol=2e-3, rtol=3e-2)

## 5. Backward: first the expert path, then the router

For each grouped projection \(C_e=A_eB_e^\mathsf{T}\), the gradients are \(dA_e=dC_eB_e\) and \(dB_e=dC_e^\mathsf{T}A_e\). The first reuses the forward grouped kernel; the second reduces over each expert's token rows.

We follow the article's `selected_expert_path_backward`: return the expert contribution to \(dX\), the selected-weight gradient \(dp\), and the three expert weight gradients. Routing weights and the weighted sum use FP32; expert GEMMs use the teaching dtype. We accumulate token input gradients in FP32 before combining the expert and router paths. The selected expert indices stay fixed.

In [ ]:
@triton.jit
def _grouped_weight_grad_kernel(
    a_ptr, dc_ptr, db_ptr, offsets_ptr,
    N: tl.constexpr, K: tl.constexpr, MAX_M: tl.constexpr,
    stride_am, stride_ak, stride_cm, stride_cn, stride_be, stride_bk, stride_bn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid = tl.program_id(0)
    tiles_k, tiles_n = tl.cdiv(K, BLOCK_K), tl.cdiv(N, BLOCK_N)
    expert = pid // (tiles_k * tiles_n)
    rem = pid % (tiles_k * tiles_n)
    tile_k, tile_n = rem // tiles_n, rem % tiles_n
    group_start = tl.load(offsets_ptr + expert)
    group_end = tl.load(offsets_ptr + expert + 1)
    ks = tile_k * BLOCK_K + tl.arange(0, BLOCK_K)
    ns = tile_n * BLOCK_N + tl.arange(0, BLOCK_N)
    acc = tl.zeros((BLOCK_K, BLOCK_N), dtype=tl.float32)
    for row_start in range(0, MAX_M, BLOCK_M):
        rows = row_start + tl.arange(0, BLOCK_M)
        a = tl.load(a_ptr + (group_start + rows)[:, None] * stride_am + ks[None, :] * stride_ak,
                    mask=(group_start + rows[:, None] < group_end) & (ks[None, :] < K), other=0.0)
        dc = tl.load(dc_ptr + (group_start + rows)[:, None] * stride_cm + ns[None, :] * stride_cn,
                     mask=(group_start + rows[:, None] < group_end) & (ns[None, :] < N), other=0.0)
        acc += tl.dot(tl.trans(a), dc)
    tl.store(db_ptr + expert * stride_be + ns[:, None] * stride_bn + ks[None, :] * stride_bk,
             tl.trans(acc).to(db_ptr.dtype.element_ty), mask=(ns[:, None] < N) & (ks[None, :] < K))

def grouped_weight_grad(a, dc, offsets, *, block_m=32, block_n=64, block_k=32):
    a, dc = a.contiguous(), dc.contiguous()
    E, K, N = offsets.numel() - 1, a.shape[1], dc.shape[1]
    db = torch.empty((E, N, K), device=a.device, dtype=a.dtype)
    max_m = max(1, int((offsets[1:] - offsets[:-1]).max().item()))
    grid = (E * math.ceil(K / block_k) * math.ceil(N / block_n),)
    _grouped_weight_grad_kernel[grid](
        a, dc, db, offsets, N, K, max_m,
        a.stride(0), a.stride(1), dc.stride(0), dc.stride(1),
        db.stride(0), db.stride(2), db.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k, num_warps=4,
    )
    return db

def grouped_mm_backward(a, b, dc, offsets):
    da = grouped_mm_contiguous(dc, b.transpose(1, 2).contiguous(), offsets)
    db = grouped_weight_grad(a, dc, offsets)
    return da, db

def silu_grad(x):
    s = torch.sigmoid(x)
    return s * (1.0 + x * (1.0 - s))

def selected_expert_path_backward(
    dout, cache, w_gate, w_up, w_down, x_shape, routing_shape
):
    (token_ids, _, pair_weight, order, offsets,
     a, gate, up, hidden, pair_out) = cache

    d_pair_out = (dout[token_ids].float() * pair_weight[:, None]).to(pair_out.dtype)
    d_pair_weight = (
        dout[token_ids].float() * pair_out.float()
    ).sum(dim=1)

    d_hidden, d_w_down = grouped_mm_backward(
        hidden, w_down, d_pair_out, offsets
    )
    d_gate = d_hidden * up * silu_grad(gate)
    d_up = d_hidden * torch.nn.functional.silu(gate)
    d_a_gate, d_w_gate = grouped_mm_backward(
        a, w_gate, d_gate, offsets
    )
    d_a_up, d_w_up = grouped_mm_backward(
        a, w_up, d_up, offsets
    )

    d_x = torch.zeros(x_shape, device=dout.device, dtype=torch.float32)
    d_x.index_add_(0, token_ids, d_a_gate.float() + d_a_up.float())

    # d_pair_weight is expert-major; scatter it back to token/top-k order.
    d_routing_flat = torch.zeros(
        routing_shape.numel(), device=dout.device, dtype=pair_weight.dtype
    )
    d_routing_flat.scatter_(0, order, d_pair_weight.to(pair_weight.dtype))
    return (
        d_x,
        d_routing_flat.reshape(routing_shape.shape),
        d_w_gate,
        d_w_up,
        d_w_down,
    )

In [ ]:
# First check the expert path with routing weights as independent inputs.
torch.manual_seed(1)
T, D, FFE, E, TOP_K = 20, 32, 48, 6, 2
x0 = torch.randn((T, D), device=DEVICE, dtype=DTYPE)
w_router0 = torch.randn((E, D), device=DEVICE, dtype=torch.float32) * 0.05
logits0 = x0.float() @ w_router0.T
selected_expert = torch.topk(logits0, TOP_K, dim=-1).indices
weight0 = logits0.gather(1, selected_expert).softmax(-1)
w0 = [torch.randn(shape, device=DEVICE, dtype=DTYPE) * 0.05
      for shape in ((E, FFE, D), (E, FFE, D), (E, D, FFE))]

x_ref = x0.clone().requires_grad_()
weight_ref = weight0.clone().requires_grad_()
w_ref = [v.clone().requires_grad_() for v in w0]
ref = moe_reference(x_ref, selected_expert, weight_ref, *w_ref)
dout = torch.randn_like(ref)
(ref.float() * dout.float()).sum().backward()

with torch.no_grad():
    out_tri, cache = selected_expert_path_forward(
        x0, selected_expert, weight0, *w0
    )
    expert_grads = selected_expert_path_backward(
        dout, cache, *w0, x0.shape, weight0
    )
check_close("expert forward", out_tri, ref)
for name, actual, expected in zip(
    ("dX expert", "dp selected weights", "dW_gate", "dW_up", "dW_down"),
    expert_grads, (x_ref.grad, weight_ref.grad, *[w.grad for w in w_ref]),
):
    check_close(name, actual, expected, atol=3e-3, rtol=3e-2)

### Continue through the selected softmax and router projection

The expert backward returned \(dp\). For the selected softmax, \(dr=p\odot(dp-\sum_j p_jdp_j)\). We scatter this into the full logit tensor, compute \(dW_r=dR^\top X\), and add \(dR\,W_r\) to the expert contribution to \(dX\).

We now connect the router to \(X\) and check the whole computation against PyTorch autograd, including \(dp\). Top-k indices remain fixed; this test covers the task-loss gradient, without a balancing loss.

In [ ]:
def selected_router_backward(
    x, w_router, topk_idx, topk_weight, d_selected_weight
):
    # Router arithmetic stays in FP32 even when expert activations use BF16.
    p = topk_weight.float()
    dp = d_selected_weight.float()
    centered = dp - (
        p * dp
    ).sum(dim=-1, keepdim=True)
    d_selected_logits = p * centered

    d_logits = torch.zeros(
        (x.shape[0], w_router.shape[0]),
        device=x.device,
        dtype=torch.float32,
    )
    d_logits.scatter_add_(1, topk_idx, d_selected_logits)

    d_w_router = d_logits.transpose(0, 1) @ x.float()
    d_x_router = d_logits @ w_router.float()
    return d_x_router, d_w_router

# Reuse x0, w0, w_router0, selected_expert, and dout from the previous test.
x_ref = x0.clone().requires_grad_()
w_router_ref = w_router0.clone().requires_grad_()
w_ref = [v.clone().requires_grad_() for v in w0]
logits_ref = x_ref.float() @ w_router_ref.T
weight_ref = logits_ref.gather(1, selected_expert).softmax(-1)
weight_ref.retain_grad()
ref = moe_reference(x_ref, selected_expert, weight_ref, *w_ref)
(ref.float() * dout.float()).sum().backward()

with torch.no_grad():
    weight_tri = (x0.float() @ w_router0.T).gather(1, selected_expert).softmax(-1)
    out_tri, cache = selected_expert_path_forward(
        x0, selected_expert, weight_tri, *w0
    )
    dx_expert, dp, dwg, dwu, dwd = selected_expert_path_backward(
        dout, cache, *w0, x0.shape, weight_tri
    )
    dx_router, dwr = selected_router_backward(
        x0, w_router0, selected_expert, weight_tri, dp
    )
    dx = (dx_expert + dx_router).to(x0.dtype)

for name, actual, expected in zip(
    ("dX expert + router", "dp selected weights", "dW_router",
     "dW_gate", "dW_up", "dW_down"),
    (dx, dp, dwr, dwg, dwu, dwd),
    (x_ref.grad, weight_ref.grad, w_router_ref.grad, *[w.grad for w in w_ref]),
):
    check_close(name, actual, expected, atol=3e-3, rtol=3e-2)

## 6. Benchmarking: separate schedule preparation from computation

We compare balanced and skewed loads with the same total of 1,024 rows, then a separate decode workload with only eight rows. Widths stay fixed.

**Wrapper time** is synchronized wall time including count reads, schedule construction, allocation, and GPU work. **Prepared time** uses CUDA events after we have prepared the schedules, host offsets, and output buffers. The per-expert loop still launches separate matmuls, so its prepared time includes gaps between those launches. These are teaching implementations, not a production-library performance comparison.


In [ ]:
def grouped_from_counts(counts, K=256, N=256):
    counts = torch.tensor(counts, device=DEVICE, dtype=torch.int32)
    offsets = torch.cat([torch.zeros(1, device=DEVICE, dtype=torch.int32), counts.cumsum(0)])
    a = torch.randn((int(counts.sum()), K), device=DEVICE, dtype=DTYPE)
    b = torch.randn((len(counts), N, K), device=DEVICE, dtype=DTYPE)
    return a, b, offsets

def loop_expert_mm(a, b, offsets):
    return torch.cat([
        a[offsets[e].item():offsets[e + 1].item()] @ b[e].transpose(0, 1)
        for e in range(b.shape[0])
    ])

def loop_expert_mm_prepared(a, b, host_offsets, out):
    for e, (start, end) in enumerate(zip(host_offsets[:-1], host_offsets[1:])):
        if start != end:
            torch.mm(a[start:end], b[e].T, out=out[start:end])
    return out

for label, counts in {
    "balanced training": [128] * 8,
    "skewed training": [768, 128, 64, 32, 16, 8, 4, 4],
    "decode-like": [2, 1, 0, 2, 1, 0, 1, 1],
}.items():
    aa, bb, oo = grouped_from_counts(counts)
    schedule = make_contiguous_schedule(torch.tensor(counts), 32, device=DEVICE)
    host_offsets = oo.cpu().tolist()
    grouped_out = torch.empty((aa.shape[0], bb.shape[1]), device=DEVICE, dtype=DTYPE)
    loop_out = torch.empty_like(grouped_out)
    grouped_prepared = lambda: grouped_mm_contiguous(aa, bb, oo, schedule=schedule, out=grouped_out)
    loop_prepared = lambda: loop_expert_mm_prepared(aa, bb, host_offsets, loop_out)
    check_close(label, grouped_prepared(), loop_prepared(), atol=5e-2)
    grouped_wall = benchmark_wall_ms(lambda: grouped_mm_contiguous(aa, bb, oo))
    loop_wall = benchmark_wall_ms(lambda: loop_expert_mm(aa, bb, oo))
    grouped_ms = benchmark_ms(grouped_prepared)
    loop_ms = benchmark_ms(loop_prepared)
    print(f"{label} | rows={sum(counts)}")
    print(f"  wrapper wall: grouped={grouped_wall:.3f} ms, loop={loop_wall:.3f} ms")
    print(f"  prepared GPU: grouped={grouped_ms:.3f} ms, loop={loop_ms:.3f} ms")


## 7. Decode: test Split-K and column-major scheduling

A decode batch can leave an expert with only one or two token rows. We isolate one expert's \(C=AB\) here so we can change the schedule without also changing routing. The kernel below is the Section 1 GEMM with two options:

- **Split-K** assigns disjoint ranges of input features to different programs. They write FP32 partial outputs, which a second kernel sums. The benchmark includes this reduction.
- **Column-major scheduling** assigns nearby program IDs to row tiles that read the same weight block. It changes neither the stored tensors nor the result, and it does not enforce execution order.

When there is only one row tile, row-major and column-major mappings are identical. We therefore use a tiny-row workload for Split-K and a separate workload with several row tiles for the ordering experiment. Neither change guarantees a speedup.

In [ ]:
@triton.jit
def _decode_matmul_kernel(
    A, B, C,
    M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
    stride_am, stride_ak, stride_bk, stride_bn,
    SPLIT_K: tl.constexpr, COLUMN_MAJOR: tl.constexpr,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid = tl.program_id(0)
    split = tl.program_id(1)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    if COLUMN_MAJOR:
        pid_m = pid % num_pid_m
        pid_n = pid // num_pid_m
    else:
        pid_m = pid // num_pid_n
        pid_n = pid % num_pid_n
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    # Each split owns whole K tiles. Tail and empty splits load zeros.
    k_tiles_per_split = tl.cdiv(tl.cdiv(K, BLOCK_K), SPLIT_K)
    first_tile = split * k_tiles_per_split
    acc = tl.zeros((BLOCK_M, BLOCK_N), tl.float32)
    for tile in range(first_tile, first_tile + k_tiles_per_split):
        ks = tile * BLOCK_K + tl.arange(0, BLOCK_K)
        a = tl.load(
            A + rows[:, None] * stride_am + ks[None, :] * stride_ak,
            mask=(rows[:, None] < M) & (ks[None, :] < K), other=0.0,
        )
        b = tl.load(
            B + ks[:, None] * stride_bk + cols[None, :] * stride_bn,
            mask=(ks[:, None] < K) & (cols[None, :] < N), other=0.0,
        )
        acc += tl.dot(a, b)
    # SPLIT_K=1 writes the final output; otherwise C is [SPLIT_K,M,N].
    tl.store(
        C + split * M * N + rows[:, None] * N + cols[None, :],
        acc.to(C.dtype.element_ty),
        mask=(rows[:, None] < M) & (cols[None, :] < N),
    )


@triton.jit
def _reduce_split_k(
    PARTIALS, OUT,
    SIZE: tl.constexpr, SPLIT_K: tl.constexpr,
    BLOCK_S: tl.constexpr, BLOCK: tl.constexpr,
):
    idx = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    splits = tl.arange(0, BLOCK_S)
    values = tl.load(
        PARTIALS + splits[:, None] * SIZE + idx[None, :],
        mask=(splits[:, None] < SPLIT_K) & (idx[None, :] < SIZE),
        other=0.0,
    )
    result = tl.sum(values, axis=0)
    tl.store(OUT + idx, result.to(OUT.dtype.element_ty), mask=idx < SIZE)


def decode_matmul(
    a, b, *, split_k=1, column_major=False,
    block_m=32, block_n=64, block_k=32, out=None, partials=None,
):
    assert a.ndim == b.ndim == 2 and a.shape[1] == b.shape[0]
    assert a.is_cuda and b.is_cuda and a.device == b.device
    assert a.dtype == b.dtype == DTYPE and split_k >= 1
    M, K = a.shape
    N = b.shape[1]
    if out is None:
        out = torch.empty((M, N), device=a.device, dtype=a.dtype)
    assert out.shape == (M, N) and out.dtype == a.dtype and out.is_contiguous()
    assert out.device == a.device
    if split_k == 1:
        target = out
    else:
        if partials is None:
            partials = torch.empty((split_k, M, N), device=a.device, dtype=torch.float32)
        assert partials.shape == (split_k, M, N) and partials.dtype == torch.float32
        assert partials.is_contiguous() and partials.device == a.device
        target = partials
    grid = (triton.cdiv(M, block_m) * triton.cdiv(N, block_n), split_k)
    _decode_matmul_kernel[grid](
        a, b, target, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        SPLIT_K=split_k, COLUMN_MAJOR=column_major,
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k, num_warps=4,
    )
    if split_k > 1:
        _reduce_split_k[(triton.cdiv(M * N, 256),)](
            target, out, M * N, split_k,
            BLOCK_S=triton.next_power_of_2(split_k), BLOCK=256,
        )
    return out


# Check tails, non-power-of-two split counts, and empty K partitions.
torch.manual_seed(2)
for M, K, N in [(1, 97, 80), (65, 33, 70)]:
    a = torch.randn((M, K), device=DEVICE, dtype=DTYPE) * 0.1
    b = torch.randn((K, N), device=DEVICE, dtype=DTYPE) * 0.1
    reference = a @ b
    for split_k in (1, 2, 3, 8):
        for column_major in (False, True):
            actual = decode_matmul(a, b, split_k=split_k, column_major=column_major)
            check_close(f"M={M}, K={K}, split={split_k}, column_major={column_major}",
                        actual, reference)

### Measure the two changes separately

We keep tile sizes and inputs fixed within each experiment. Output and partial buffers are allocated before timing; each Split-K measurement includes both the partial GEMM and its reduction. Warmup excludes first-use compilation.

The second workload has multiple row tiles per expert so the two orders can differ. Its result depends on the GPU's L2 size and scheduling; it does not predict the speed of an expert receiving only one row.

In [ ]:
def prepared_decode_runner(a, b, *, split_k=1, column_major=False):
    M, N = a.shape[0], b.shape[1]
    out = torch.empty((M, N), device=a.device, dtype=a.dtype)
    partials = None if split_k == 1 else torch.empty(
        (split_k, M, N), device=a.device, dtype=torch.float32
    )
    return lambda: decode_matmul(
        a, b, split_k=split_k, column_major=column_major,
        out=out, partials=partials,
    )


torch.manual_seed(3)
# Few output tiles, with a long reduction dimension.
a = torch.randn((1, 4096), device=DEVICE, dtype=DTYPE) * 0.1
b = torch.randn((4096, 1024), device=DEVICE, dtype=DTYPE) * 0.1
reference = a @ b
for split_k in (1, 2, 4, 8):
    run = prepared_decode_runner(a, b, split_k=split_k)
    check_close(f"Split-K={split_k}", run(), reference)
    print(f"Split-K={split_k}: {benchmark_ms(run):.3f} ms (including reduction)")

# Several row tiles can read the same weight block through L2.
a = torch.randn((256, 2048), device=DEVICE, dtype=DTYPE) * 0.1
b = torch.randn((2048, 4096), device=DEVICE, dtype=DTYPE) * 0.1
reference = a @ b
for column_major in (False, True):
    run = prepared_decode_runner(a, b, column_major=column_major)
    order = "column-major" if column_major else "row-major"
    check_close(order, run(), reference)
    print(f"{order}: {benchmark_ms(run):.3f} ms")

## 8. DeepGEMM: call the official BF16 grouped kernel

On a compatible machine, we call `m_grouped_bf16_gemm_nt_contiguous(a, b, d, grouped_layout)`. The inputs are \(A:[M,K]\), weights \(B:[E,N,K]\), and one expert ID per packed row. Expert segments must satisfy the library's alignment, so we query that alignment below.

This optional example follows DeepGEMM revision `559d79fb6994a58b8a15b4b93bf13ccc16edf247`. It needs an SM90 or SM100 GPU, PyTorch 2.1+, the matching installed API, and a CUDA Toolkit compiler: at least 12.3 for SM90 or 12.9 for SM100. A usual T4/L4 Colab runtime skips it. The PyTorch CUDA runtime version alone does not tell us which toolkit will compile the kernels.

See the [pinned requirements](https://github.com/deepseek-ai/DeepGEMM/tree/559d79fb6994a58b8a15b4b93bf13ccc16edf247#requirements) and [BF16 tests](https://github.com/deepseek-ai/DeepGEMM/blob/559d79fb6994a58b8a15b4b93bf13ccc16edf247/tests/test_bf16.py).

In [ ]:
def deep_gemm_status():
    import os
    import re
    import shutil
    from pathlib import Path
    from packaging.version import Version

    if CAPABILITY[0] not in (9, 10):
        return False, f"GPU is sm_{CAPABILITY[0]}{CAPABILITY[1]}; this example targets SM90/SM100"
    if Version(torch.__version__.split("+")[0]) < Version("2.1"):
        return False, f"DeepGEMM needs PyTorch >= 2.1; found {torch.__version__}"
    # Prefer the compiler DeepGEMM was explicitly configured to use.
    nvcc = os.environ.get("DG_JIT_NVCC_COMPILER")
    if not nvcc:
        from torch.utils.cpp_extension import CUDA_HOME
        nvcc = str(Path(CUDA_HOME) / "bin" / "nvcc") if CUDA_HOME else shutil.which("nvcc")
    if not nvcc:
        return False, "nvcc is missing; the CUDA Toolkit is required for JIT compilation"
    required = (12, 9) if CAPABILITY[0] == 10 else (12, 3)
    try:
        version_text = subprocess.check_output([nvcc, "--version"], text=True, stderr=subprocess.STDOUT)
    except (OSError, subprocess.CalledProcessError) as exc:
        return False, f"cannot run nvcc: {exc}"
    match = re.search(r"release\s+(\d+)\.(\d+)", version_text)
    if not match:
        return False, "cannot parse the CUDA Toolkit version from nvcc --version"
    toolkit = tuple(map(int, match.groups()))
    if toolkit < required:
        return False, f"SM{CAPABILITY[0]}0 needs CUDA Toolkit >= {required}; nvcc reports {toolkit}"
    if importlib.util.find_spec("deep_gemm") is None:
        return False, "deep_gemm is not installed; build the pinned official repository"
    try:
        import deep_gemm
    except Exception as exc:
        return False, f"deep_gemm import failed: {exc}"
    required_api = ("m_grouped_bf16_gemm_nt_contiguous",
                    "get_mk_alignment_for_contiguous_layout")
    missing = [name for name in required_api if not callable(getattr(deep_gemm, name, None))]
    if missing:
        return False, "installed DeepGEMM lacks: " + ", ".join(missing)
    return True, deep_gemm


DG_OK, DG = deep_gemm_status()
print("DeepGEMM status:", "available" if DG_OK else DG)

In [ ]:
if not DG_OK:
    print("Skipping official DeepGEMM call.")
else:
    alignment = DG.get_mk_alignment_for_contiguous_layout()
    rows_per_expert = triton.cdiv(128, alignment) * alignment
    num_experts, K, N = 4, 256, 192
    a = torch.randn((num_experts * rows_per_expert, K), device=DEVICE, dtype=torch.bfloat16)
    b_nt = torch.randn((num_experts, N, K), device=DEVICE, dtype=torch.bfloat16)
    d = torch.empty((a.shape[0], N), device=DEVICE, dtype=torch.bfloat16)
    grouped_layout = torch.repeat_interleave(
        torch.arange(num_experts, device=DEVICE, dtype=torch.int32),
        rows_per_expert,
    )
    DG.m_grouped_bf16_gemm_nt_contiguous(a, b_nt, d, grouped_layout)
    reference = torch.cat([
        a[e * rows_per_expert:(e + 1) * rows_per_expert] @ b_nt[e].transpose(0, 1)
        for e in range(num_experts)
    ])
    check_close("official DeepGEMM BF16 grouped GEMM", d, reference, atol=5e-2, rtol=5e-2)

## Takeaways

We can now follow token-major routing → expert-major packed rows → grouped GEMM → token-major combine, and check the expert and router gradients separately.

For decode, Split-K adds parallel work along the reduction dimension and pays for merging partial results. Column-major scheduling can improve weight reuse through L2 when an expert has multiple row tiles.

For the article's differentiable multi-GPU path, run the [EP script](https://github.com/G-U-N/G-U-N.github.io/blob/master/blogs/code/moe_ep_hopper.py) from the repository root:

```bash
torchrun --standalone --nproc-per-node=4 blogs/code/moe_ep_hopper.py \
  --backend torch-triton --phase train --num-experts 8 --top-k 2 --check
```

That script uses one full-world EP group. With the official libraries installed, `--backend both --phase decode --check` compares the PyTorch/Triton and DeepEP/DeepGEMM forward pipelines. The official backend comparison covers forward only.